### **Project Title: Space-Time Clustering of Mobility Patterns and Air Quality Hotspots**

Install Required Packages and import Libraries

In [3]:
!pip install pygeohash
!pip install folium
!pip install uszipcode
!pip install geopandas
!pip install datascience


In [5]:
from datascience import *
import pandas as pd
import geopandas as gpd
import pygeohash as gh
import numpy as np
from shapely.geometry import Polygon
from shapely.geometry import Point
from shapely import wkt
from geopandas.tools import sjoin
import matplotlib.pyplot as plt

plt.style.use('fivethirtyeight')


## **1. Data Exploration & Cleaning**

Steps:
Inspect each dataset:

*   Check dtypes format , timestamp formats, intial statistics, spatial coordinates, and column consistency.

*   Remove erroneous coordinates like (0,0).

*   Convert timestamps to a consistent format.

*   Clean and handle missing/null values.


Read CSV files Automated_Traffic_Volume_Counts, NYC_pm, and gejson file nyc_polygon

In [38]:
# download data_set_link="https://www.kaggle.com/datasets/aadimator/nyc-automated-traffic-volume-counts/data"

#Read the CSV file containing PM sensors readings and AQ file
PM_data = pd.read_csv('https://raw.githubusercontent.com/Dr-Isam-ALJAWARNEH/fds-project-space-time-clustering-for-aq/refs/heads/main/Datasets/NYC_PM.csv',index_col=False)
Automated_Traffic_Volume_Counts = pd.read_csv('Automated_Traffic_Volume_Counts.csv')


#Read the GeoJSON file containing neighborhood boundaries into a GeoDataFrame
nyc_neighborhoods = gpd.read_file('https://raw.githubusercontent.com/Dr-Isam-ALJAWARNEH/fds-project-space-time-clustering-for-aq/refs/heads/main/Datasets/nyc_polygon.geojson')


In [39]:
#Exploring data and their data type
print("----------------------PM_data-----------------------")
print("PM_data",PM_data.shape)
print(PM_data.describe())
print(PM_data.info())

print("----------Automated_Traffic_Volume_Counts-----------")
print("Automated_Traffic_Volume_Counts",Automated_Traffic_Volume_Counts.shape)
print(Automated_Traffic_Volume_Counts.describe())
print(Automated_Traffic_Volume_Counts.info())

print("-----------------nyc_neighborhoods------------------")
print("nyc_neighborhoods",nyc_neighborhoods.shape)
print(nyc_neighborhoods.describe())
print(nyc_neighborhoods.info())

----------------------PM_data-----------------------
PM_data (118765, 33)
               time       latitude      longitude           bin0  \
count  1.187650e+05  118765.000000  118765.000000  118765.000000   
mean   1.580374e+09      40.825551     -73.891823      44.660321   
std    5.347413e+05       0.029206       0.025815     151.277083   
min    1.579562e+09      40.008129     -74.009636       0.000000   
25%    1.579806e+09      40.815228     -73.898399       9.000000   
50%    1.580330e+09      40.821003     -73.890045      21.000000   
75%    1.580824e+09      40.846992     -73.872032      57.000000   
max    1.581927e+09      40.913586     -73.795425   18441.000000   

                bin1           bin2           bin3           bin4  \
count  118765.000000  118765.000000  118765.000000  118765.000000   
mean        8.255429       2.926519       1.033183       0.985930   
std        77.556461      72.625702      71.675580      71.659975   
min         0.000000       0.000000  

In [40]:
#uniform the data and clean them

#remove unnecessary coulmns 
#remove erroneous coordinates (0,0)
#Convert the Unix epoch time column to datetime and set it as a new column


#######PM_data######
column_list=['bin0','bin1','bin2','bin3','bin4','bin5','bin6','bin7','bin8','bin9','bin10','bin11','bin12','bin13','bin14','bin15','bin16','bin17','bin18','bin19','bin20','bin21','bin22','bin23','pm1','pm10']
for i in column_list:
    if i in PM_data.columns:
       PM_data = PM_data.drop(columns=[i])


PM_data = \
PM_data[(PM_data ['latitude']!=0) & \
       (PM_data ['longitude'] !=0)]

PM_data ['datetime'] = pd.to_datetime(PM_data['time'],unit='s')


print("PM_data time range:")
print("Oldest:", PM_data['datetime'].min())
print("Newest:", PM_data['datetime'].max())
PM_data.head(5)



PM_data time range:
Oldest: 2020-01-20 23:11:00
Newest: 2020-02-17 08:10:00


,SensorID,time,latitude,longitude,temperature,humidity,pm25,datetime
0,NYCP1_01A,1579618560,40.847183,-73.870087,16.3,15.2,5.91,2020-01-21 14:56:00
1,NYCP1_01A,1579618560,40.847183,-73.870094,16.2,15.1,1.18,2020-01-21 14:56:00
2,NYCP1_01A,1579618560,40.847179,-73.870094,16.1,15.1,0.76,2020-01-21 14:56:00
3,NYCP1_01A,1579618560,40.847179,-73.870094,16.1,15.2,4.48,2020-01-21 14:56:00
4,NYCP1_01A,1579618560,40.847179,-73.870094,16.0,15.2,5.77,2020-01-21 14:56:00


In [54]:
#######nyc_neighborhoods######
nyc_neighborhoods.head(5)


,neighborhood,boroughCode,borough,@id,geometry
0,Allerton,2,Bronx,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-73.849 40.872, -73.846 40.87, -73.8..."
1,Alley Pond Park,4,Queens,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-73.743 40.739, -73.744 40.739, -73...."
2,Arden Heights,5,Staten Island,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-74.17 40.561, -74.17 40.561, -74.16..."
3,Arlington,5,Staten Island,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-74.16 40.641, -74.16 40.641, -74.16..."
4,Arrochar,5,Staten Island,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-74.061 40.593, -74.061 40.593, -74...."


In [42]:
########Automated_Traffic_Volume_Counts#######

Automated_Traffic_Volume_Counts['datetime'] = pd.to_datetime(Automated_Traffic_Volume_Counts[['Yr', 'M', 'D', 'HH', 'MM']].rename(columns={
    'Yr': 'year', 'M': 'month', 'D': 'day', 'HH': 'hour', 'MM': 'minute'}))
ATVC = Automated_Traffic_Volume_Counts

drop_list= ['RequestID','Yr','M','D','HH','MM','SegmentID','street','fromSt','toSt','Direction']
for i in drop_list:
    if i in ATVC.columns:
        ATVC = ATVC.drop(columns=[i])

print("Automated_Traffic_Volume_Counts time range:")
print("Oldest datetime:", ATVC['datetime'].min())
print("Newest datetime:", ATVC['datetime'].max())
ATVC.head(5)



Automated_Traffic_Volume_Counts time range:
Oldest datetime: 2000-01-01 00:15:00
Newest datetime: 2020-11-22 23:45:00


,Boro,Vol,WktGeom,datetime
0,Queens,9,POINT (1052296.600156678 199785.26932711253),2015-06-23 23:30:00
1,Staten Island,6,POINT (942668.0589509147 171441.21296926),2015-09-14 04:15:00
2,Bronx,85,POINT (1016508.0034050211 235221.59092266942),2017-10-19 04:30:00
3,Brooklyn,168,POINT (992925.4316054962 184116.82855457635),2017-11-07 18:30:00
4,Manhattan,355,POINT (1004175.9505178436 247779.63624949602),2017-11-03 22:00:00


In [52]:
# # Filter the DataFrame of ATVC to fit with PM_data range: 
# Define start and end date
start_date = '2020-01-20 23:11:00'
end_date = '2020-02-17 08:10:00'

ATVC = ATVC[(ATVC['datetime'] >= start_date) & (ATVC['datetime'] <= end_date)]
print(ATVC.shape)
print("Oldest datetime:", ATVC['datetime'].min())
print("Newest datetime:", ATVC['datetime'].max())
ATVC.head(5)


(15690, 4)
Oldest datetime: 2020-01-20 23:15:00
Newest datetime: 2020-02-17 08:00:00


,Boro,Vol,WktGeom,datetime
348,Staten Island,229,POINT (948549.0884921695 148767.7228358149),2020-01-25 14:15:00
1029,Staten Island,116,POINT (948899.8883570207 149112.346113424),2020-01-25 20:00:00
5179,Manhattan,28,POINT (999848.3093221203 247759.78308722837),2020-02-09 08:00:00
6259,Staten Island,231,POINT (948912.7573158939 149574.99331751716),2020-01-25 11:00:00
6344,Staten Island,4,POINT (948899.8883570207 149112.346113424),2020-01-28 04:00:00


Convert WktGeom to Usable Coordinates in EPSG:4326 format instead of EPSG:2263

In [69]:
# Make a proper copy first
ATVC = ATVC.copy()

# Convert WKT text to geometry (Parsed WktGeom into real geometry objects:)
ATVC['geometry'] = ATVC['WktGeom'].apply(wkt.loads)

# Create a GeoDataFrame (Created a GeoDataFrame with correct original CRS:)
ATVC_gdf = gpd.GeoDataFrame(ATVC, geometry='geometry', crs='EPSG:2263')  # Assuming NY State Plane

# Convert to lat/lon (WGS84) (Reprojected to WGS84 (lat/lon):)
ATVC_gdf = gdf.to_crs('EPSG:4326')  # Now coordinates will be in lat/lon
ATVC_gdf.head(5)



,Boro,Vol,WktGeom,datetime,geometry
348,Staten Island,229,POINT (948549.0884921695 148767.7228358149),2020-01-25 14:15:00,POINT (-74.129 40.575)
1029,Staten Island,116,POINT (948899.8883570207 149112.346113424),2020-01-25 20:00:00,POINT (-74.127 40.576)
5179,Manhattan,28,POINT (999848.3093221203 247759.78308722837),2020-02-09 08:00:00,POINT (-73.944 40.847)
6259,Staten Island,231,POINT (948912.7573158939 149574.99331751716),2020-01-25 11:00:00,POINT (-74.127 40.577)
6344,Staten Island,4,POINT (948899.8883570207 149112.346113424),2020-01-28 04:00:00,POINT (-74.127 40.576)


## **2. Data Integration**

Aggregate all datasets into a common spatial grid and temporal resolution.

Spatial Aggregation:

*    Create geohash cells over NYC.
*    Spatially join GPS/mobility data and AQ data to these grid cells.

Temporal Aggregation:
*    Resample time to a fixed interval.
*    For each grid cell & time slice, calculate:

      *    Mobility: Count of trips, speed, or density.

      *    Air Quality: Mean PM2.5, humidity, and temprature.

🗺 Spatial Grid:

In [182]:
#Set configuration
geohash_precision= 4
#Generate Geohash for each tuple (long,lat)
#Ensure both GeoDataFrames are in the same CRS
ATVC_gdf = ATVC_gdf.to_crs("EPSG:4326")

########ATVC_gdf#######

# Extract lat/lon and compute geohash (Generate accurate geohash from lat/lon:)
ATVC_gdf['geohash'] = ATVC_gdf['geometry'].apply(lambda geom: gh.encode(geom.y, geom.x, precision=geohash_precision))
ATVC_gdf.head(5)

,Boro,Vol,WktGeom,datetime,geometry,geohash,time_bin
348,Staten Island,229,POINT (948549.0884921695 148767.7228358149),2020-01-25 14:15:00,POINT (-74.129 40.575),dr5q,2020-01-25 14:00:00
1029,Staten Island,116,POINT (948899.8883570207 149112.346113424),2020-01-25 20:00:00,POINT (-74.127 40.576),dr5q,2020-01-25 20:00:00
5179,Manhattan,28,POINT (999848.3093221203 247759.78308722837),2020-02-09 08:00:00,POINT (-73.944 40.847),dr72,2020-02-09 08:00:00
6259,Staten Island,231,POINT (948912.7573158939 149574.99331751716),2020-01-25 11:00:00,POINT (-74.127 40.577),dr5q,2020-01-25 11:00:00
6344,Staten Island,4,POINT (948899.8883570207 149112.346113424),2020-01-28 04:00:00,POINT (-74.127 40.576),dr5q,2020-01-28 04:00:00


In [184]:
#######PM_data######
PM_gdf = gpd.GeoDataFrame(PM_data, geometry=gpd.points_from_xy(PM_data['longitude'], PM_data['latitude']), crs='EPSG:4326')
PM_gdf = PM_gdf.to_crs("EPSG:4326")
PM_gdf['geohash']=PM_gdf.apply(lambda x: gh.encode(x.latitude,x.longitude,precision=geohash_precision),axis=1)
PM_gdf.head(5)

,SensorID,time,latitude,longitude,temperature,humidity,pm25,datetime,geohash,time_bin,geometry
0,NYCP1_01A,1579618560,40.847183,-73.870087,16.3,15.2,5.91,2020-01-21 14:56:00,dr72,2020-01-21 14:00:00,POINT (-73.87 40.847)
1,NYCP1_01A,1579618560,40.847183,-73.870094,16.2,15.1,1.18,2020-01-21 14:56:00,dr72,2020-01-21 14:00:00,POINT (-73.87 40.847)
2,NYCP1_01A,1579618560,40.847179,-73.870094,16.1,15.1,0.76,2020-01-21 14:56:00,dr72,2020-01-21 14:00:00,POINT (-73.87 40.847)
3,NYCP1_01A,1579618560,40.847179,-73.870094,16.1,15.2,4.48,2020-01-21 14:56:00,dr72,2020-01-21 14:00:00,POINT (-73.87 40.847)
4,NYCP1_01A,1579618560,40.847179,-73.870094,16.0,15.2,5.77,2020-01-21 14:56:00,dr72,2020-01-21 14:00:00,POINT (-73.87 40.847)


⏱ Temporal Resolution:
* Round timestamps to 1hour for alignment. 

🧮 Aggregation:
For each spatial cell (geohash) and time interval (1hour):
* sum total Traffic_Volume_Counts, using the Vol for each (geohash, time_bin).
* Calculate mean PM2.5 , humidity, and temperature for each (geohash, time_bin).

In [186]:
#######PM_data#######
# Round the datetime to 20-minute intervals
PM_gdf['time_bin'] = PM_gdf['datetime'].dt.floor('1h')

# Group by geohash and the time_bin column
PM_aggregated = PM_gdf.groupby(['geohash', 'time_bin']).agg({
    'temperature': 'mean',
    'humidity': 'mean',
    'pm25': 'mean',}).reset_index()
print("PM_aggregated",PM_aggregated.shape)
PM_aggregated.head(5)

PM_aggregated (391, 5)


,geohash,time_bin,temperature,humidity,pm25
0,dr57,2020-02-05 13:00:00,5.400000,70.600000,0.280000
1,dr57,2020-02-05 15:00:00,9.300000,46.100000,0.000000
2,dr5r,2020-01-28 17:00:00,7.082979,45.977660,2.529745
3,dr5r,2020-01-28 18:00:00,7.383754,42.807413,1.934590
4,dr5r,2020-01-28 19:00:00,6.940644,44.138431,1.747203


In [189]:
#######ATVC_gdf#######

# Round datetime to 1-hour bins
ATVC_gdf['time_bin'] = ATVC_gdf['datetime'].dt.floor('1h')

# Group by geohash and time_bin, summing traffic volume
traffic_agg = ATVC_gdf.groupby(['geohash', 'time_bin'])[['Vol']].sum().reset_index()
traffic_agg.head(5)


,geohash,time_bin,Vol
0,dr5q,2020-01-25 00:00:00,2314
1,dr5q,2020-01-25 01:00:00,1285
2,dr5q,2020-01-25 02:00:00,787
3,dr5q,2020-01-25 03:00:00,634
4,dr5q,2020-01-25 04:00:00,509


Merge with PM_aggregated

In [178]:
# Join traffic volume with PM data
merged_df = pd.merge(traffic_agg, PM_aggregated, on=['geohash', 'time_bin'], how='inner')
print("Merged shape:", merged_df.shape)
merged_df.head(94)


Merged shape: (94, 6)


,geohash,time_bin,Vol,temperature,humidity,pm25
0,dr5,2020-01-28 17:00:00,12050,7.082979,45.977660,2.529745
1,dr5,2020-01-28 18:00:00,11130,7.383754,42.807413,1.934590
2,dr5,2020-01-28 19:00:00,8516,6.940644,44.138431,1.747203
3,dr7,2020-02-03 03:00:00,34,14.800000,43.100000,1.400000
4,dr7,2020-02-03 13:00:00,382,6.899322,90.435254,2.804881
...,...,...,...,...,...,...
89,dr7,2020-02-09 12:00:00,404,-2.700000,86.800000,1.320000
90,dr7,2020-02-09 16:00:00,687,9.000000,45.800000,0.150000
91,dr7,2020-02-09 18:00:00,641,6.950000,60.050000,0.355000
92,dr7,2020-02-09 20:00:00,371,3.500000,76.150000,2.385000


In [173]:
print(PM_aggregated['time_bin'].min(), PM_aggregated['time_bin'].max())
print(traffic_agg['time_bin'].min(), traffic_agg['time_bin'].max())

print(set(PM_aggregated['geohash']).intersection(set(traffic_agg['geohash'])))

2020-01-20 23:00:00 2020-02-17 08:00:00
2020-01-20 23:00:00 2020-02-17 08:00:00
{'dr7', 'dr5'}


In [175]:
print("Unique PM geohashes:", sorted(PM_aggregated['geohash'].unique()))
print("Unique Traffic geohashes:", sorted(traffic_agg['geohash'].unique()))

Unique PM geohashes: ['dr5', 'dr7']
Unique Traffic geohashes: ['dr5', 'dr7']


the time_bin Geohash spatial join worked only with geohash precesion of 4 (only 2 geohashes in commomn), where 5 and 6 precesions not working

trying to do the following: Spatially Link Each Point to a Neighborhood and aggregate by:

neighborhood + time_bin

Mean for PM2.5, temperature, humidity

Sum for traffic volume



In [180]:
# Ensure everything is in EPSG:4326
PM_gdf = PM_gdf.to_crs("EPSG:4326")
ATVC_gdf = ATVC_gdf.to_crs("EPSG:4326")
nyc_neighborhoods = nyc_neighborhoods.to_crs("EPSG:4326")

# Spatial join points to neighborhoods
PM_neigh = gpd.sjoin(PM_gdf, nyc_neighborhoods, how="inner", predicate='within')
ATVC_neigh = gpd.sjoin(ATVC_gdf, nyc_neighborhoods, how="inner", predicate='within')

# Round timestamps
PM_neigh['time_bin'] = PM_neigh['datetime'].dt.floor('1h')
ATVC_neigh['time_bin'] = ATVC_neigh['datetime'].dt.floor('1h')

# Aggregate
PM_agg = PM_neigh.groupby(['neighborhood', 'time_bin']).agg({
    'temperature': 'mean',
    'humidity': 'mean',
    'pm25': 'mean'
}).reset_index()

ATVC_agg = ATVC_neigh.groupby(['neighborhood', 'time_bin']).agg({
    'Vol': 'sum'
}).reset_index()

# Merge both aggregations
final_agg = pd.merge(PM_agg, ATVC_agg, on=['neighborhood', 'time_bin'], how='inner')
final_agg.head(20)

,neighborhood,time_bin,temperature,humidity,pm25,Vol


In [196]:
print (ATVC_agg.shape)
print("Unique neighborhood:", sorted(ATVC_agg['neighborhood'].unique()))
ATVC_agg.head(5)

(826, 3)
Unique neighborhood: ['Far Rockaway', 'Latourette Park', 'Lighthouse Hill', 'Richmondtown', 'Washington Heights', 'Williamsburg']


,neighborhood,time_bin,Vol
0,Far Rockaway,2020-01-20 23:00:00,91
1,Far Rockaway,2020-01-21 00:00:00,66
2,Far Rockaway,2020-01-21 01:00:00,37
3,Far Rockaway,2020-01-21 02:00:00,32
4,Far Rockaway,2020-01-21 03:00:00,32


In [198]:
print (PM_agg.shape)
print("Unique neighborhood:", sorted(PM_agg['neighborhood'].unique()))
PM_agg.head(5)

(824, 5)
Unique neighborhood: ['Allerton', 'Astoria', 'Belmont', 'Bronx Park', 'Bronxdale', 'Civic Center', 'Claremont Village', 'Concourse', 'Concourse Village', 'Country Club', 'Crotona Park', 'DUMBO', 'Ditmars Steinway', 'Downtown Brooklyn', 'East Elmhurst', 'East Harlem', 'East Morrisania', 'East Village', 'Financial District', 'Fordham', 'Harlem', 'Highbridge', 'Hunts Point', 'Kingsbridge', 'Kips Bay', 'Long Island City', 'Longwood', 'Lower East Side', 'Midtown', 'Morris Heights', 'Morris Park', 'Morrisania', 'Mott Haven', 'Mount Eden', 'Mount Hope', 'Murray Hill', 'Norwood', 'Olinville', 'Parkchester', 'Pelham Bay', 'Pelham Bay Park', 'Pelham Gardens', 'Port Morris', "Randall's Island", 'Schuylerville', 'Soundview', 'Stuyvesant Town', 'Sunnyside', 'Tremont', 'Tribeca', 'Two Bridges', 'Unionport', 'University Heights', 'Upper East Side', 'Van Cortlandt Park', 'Van Nest', 'Wakefield', 'West Farms', 'Woodlawn', 'Woodside']


,neighborhood,time_bin,temperature,humidity,pm25
0,Allerton,2020-01-27 15:00:00,7.300000,54.625000,4.208333
1,Astoria,2020-01-28 17:00:00,6.739381,46.988053,1.547168
2,Astoria,2020-01-28 18:00:00,8.300000,40.712500,1.953750
3,Astoria,2020-01-28 19:00:00,6.984375,43.782812,0.311406
4,Belmont,2020-01-30 18:00:00,3.838095,37.080952,2.852857


In [200]:
print(set(PM_agg['neighborhood']).intersection(set(ATVC_agg['neighborhood'])))

set()
